In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata>=0.12.11",
#     "dask==2026.1.1",
#     "ome-zarr>=0.16.0",
#     "scanpy>=1.12.1",
#     "spatialdata==0.7.3a1",
#     "zarr>=3.1.6",
# ]
#
# [tool.uv]
# exclude-newer = "2026-04-30T13:20:36.358035822+02:00"
# prerelease = "allow"
# override-dependencies = ["dask==2026.1.1"]
# ///


In [2]:
from pathlib import Path
import zipfile
import zarr

import numpy as np
import pandas as pd
from scipy import sparse

import anndata as ad
from spatialdata import SpatialData
from spatialdata.models import Image2DModel, ShapesModel, TableModel

/home/klaus/ws/zarr-test-datasets/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
num_patients = 12

disease_color = 'rgb(250, 202, 192)'
disease_color_arr = disease_color.replace("rgb(", "").replace(")", "").split(", ")

healthy_color = 'rgb(169, 219, 192)'
healthy_color_arr = healthy_color.replace("rgb(", "").replace(")", "").split(", ")

In [4]:
# Zone definitions: name -> cell-type probability distribution
# Order: ["T cell", "B cell", "Myeloid", "Epithelial", "Stromal"]
ZONE_NAMES = ["Stroma", "Vessel", "Epithelium"]
ZONE_CELL_TYPE_PROBS = {
    "Stroma": [0.20, 0.10, 0.20, 0.10, 0.40],
    "Vessel": [0.35, 0.25, 0.25, 0.05, 0.10],
    "Epithelium": [0.20, 0.10, 0.10, 0.50, 0.10],
}


def _make_gene_programs(cell_types_arr, global_scale):
    """Build a (n_cells, 50) count matrix from 5 co-expression programs."""
    n = len(cell_types_arr)

    # Program A - Housekeeping (genes 1-10): low, uniform across all cells
    prog_a = np.random.poisson(lam=1.0 * global_scale, size=(n, 10))

    # Program B - Cell Cycle (genes 11-20): 30% of cells randomly "cycling"
    cycling = np.random.rand(n) < 0.30
    lam_b = np.where(cycling, 5.0, 0.2)
    prog_b = np.random.poisson(lam=lam_b[:, None] * np.ones((n, 10)))

    # Program C - T-cell specific (genes 21-30)
    is_tcell = (cell_types_arr == "T cell").astype(float)
    lam_c = np.where(is_tcell, 4.0 * global_scale, 0.1)
    prog_c = np.random.poisson(lam=lam_c[:, None] * np.ones((n, 10)))

    # Program D - B-cell specific (genes 31-40)
    is_bcell = (cell_types_arr == "B cell").astype(float)
    lam_d = np.where(is_bcell, 4.0 * global_scale, 0.1)
    prog_d = np.random.poisson(lam=lam_d[:, None] * np.ones((n, 10)))

    # Program E - Myeloid/Stromal (genes 41-50)
    is_myeloid_stromal = np.isin(cell_types_arr, ["Myeloid", "Stromal"]).astype(float)
    lam_e = np.where(is_myeloid_stromal, 3.5 * global_scale, 0.2)
    prog_e = np.random.poisson(lam=lam_e[:, None] * np.ones((n, 10)))

    counts = np.hstack([prog_a, prog_b, prog_c, prog_d, prog_e]).astype(float)

    # Dropout: per-cell rate uniform in [0.20, 0.40]
    dropout_rates = np.random.uniform(0.20, 0.40, size=n)
    dropout_mask = np.random.rand(n, 50) < dropout_rates[:, None]
    counts[dropout_mask] = 0.0

    return counts.astype(np.int32)


def _draw_synthetic_nuclei(img_data, coords, nucleus_radius=2):
    """Overlay small nucleus circles centered at spot coordinates."""
    yy, xx = np.ogrid[:img_data.shape[1], :img_data.shape[2]]
    for x, y in coords:
        nucleus_mask = (xx - x) ** 2 + (yy - y) ** 2 <= nucleus_radius ** 2

        # Darker center with slight channel variation for a natural look.
        nucleus_color = np.array([
            np.random.randint(45, 75),
            np.random.randint(45, 80),
            np.random.randint(60, 95),
        ], dtype=np.float32)

        img_data[:, nucleus_mask] = nucleus_color[:, None]


def generate_synthetic_zarr(path="synthetic_study.zarr"):
    cell_types = ["T cell", "B cell", "Myeloid", "Epithelial", "Stromal"]
    images = {}
    shapes = {}
    tables = []

    for i in range(1, num_patients + 1):
        # Zero-padding prevents prefix collisions such as patient_1 vs patient_10.
        sample_id = f"patient_{i:02d}"
        shape_key = sample_id
        image_key = f"{shape_key}_hires"
        status = "disease" if i % 3 != 0 else "healthy"
        batch = "Batch_01" if i % 2 == 1 else "Batch_02"
        patient_age = int(np.random.randint(20, 80))
        global_scale = 1.3 if status == "disease" else 0.8

        # 1. Zone-biased spot coordinates (3 Gaussian clusters, one per zone)
        zone_centers = np.random.randint(10, 90, size=(3, 2))
        spots_per_zone = [34, 33, 33]  # sums to 100
        coords_list, zone_labels = [], []
        for z_idx, (zone_name, n_spots) in enumerate(zip(ZONE_NAMES, spots_per_zone)):
            cx, cy = zone_centers[z_idx]
            zc = np.random.normal(loc=[cx, cy], scale=12, size=(n_spots, 2))
            zc = np.clip(zc, 0, 99).astype(int)
            coords_list.append(zc)
            zone_labels.extend([zone_name] * n_spots)
        coords = np.vstack(coords_list)

        shapes[shape_key] = ShapesModel.parse(
            coords,
            geometry=0,
            radius=np.full(100, 1.0),
            index=[f"{shape_key}_{j}" for j in range(100)],
        )

        # 2. Generate grainy tissue image and overlay synthetic nuclei.
        base_color = np.array(
            disease_color_arr if status == "disease" else healthy_color_arr,
            dtype=np.float32,
        )
        img_data = np.full((3, 100, 100), base_color[:, None, None], dtype=np.float32)

        # Reduced grain strength from scale=15 to scale=8.
        img_data += np.random.normal(loc=0, scale=8, size=img_data.shape)
        _draw_synthetic_nuclei(img_data, coords, nucleus_radius=3)

        img_data = np.clip(img_data, 0, 255).astype(np.uint8)
        images[image_key] = Image2DModel.parse(img_data, dims=("c", "y", "x"))

        # 3. Zone-biased cell types
        cell_types_arr = np.array([
            np.random.choice(cell_types, p=ZONE_CELL_TYPE_PROBS[zone])
            for zone in zone_labels
        ])

        # 4. Gene programs (patient status x cell type hierarchy) + dropout
        counts = _make_gene_programs(cell_types_arr, global_scale)

        obs = pd.DataFrame({
            "region": pd.Categorical([shape_key] * 100),
            "patient_status": pd.Categorical([status] * 100),
            "batch": pd.Categorical([batch] * 100),
            "patient_age": patient_age,
            "instance_id": np.arange(100),
            "tissue_zone": pd.Categorical(zone_labels),
            "cell_type": pd.Categorical(cell_types_arr),
            "n_counts": counts.sum(axis=1),
            "n_genes": (counts > 0).sum(axis=1),
            "quality_score": np.random.uniform(0.0, 1.0, size=100),
        })
        obs.index = [f"{sample_id}_{j}" for j in range(100)]

        var = pd.DataFrame(index=[f"Gene {j}" for j in range(1, 51)])
        ann = ad.AnnData(X=sparse.csc_matrix(counts), obs=obs, var=var)
        ann.obsm["spatial"] = coords.astype(float)
        ann.obsm["X_umap"] = np.random.normal(loc=0.0, scale=1.0, size=(100, 2))
        tables.append(ann)

    # 5. Combine and write
    merged_table = ad.concat(tables)
    merged_table.X = sparse.csc_matrix(merged_table.X)

    sdata = SpatialData(
        images=images,
        shapes=shapes,
        tables={
            "table": TableModel.parse(
                merged_table,
                region=list(shapes.keys()),
                region_key="region",
                instance_key="instance_id",
            )
        },
    )

    sdata.write(path, overwrite=True)
    # does not make a difference to run consolidate_metadata
    # store = zarr.storage.LocalStore(path)
    # result = zarr.consolidate_metadata(store)


    # zip_path = f"{Path(path)}.zip"
    # with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
    #     for child in Path(path).iterdir():
    #         if child.is_file():
    #             archive.write(child, arcname=child.name)
    #         elif child.is_dir():
    #             for file_path in child.rglob("*"):
    #                 if file_path.is_file():
    #                     archive.write(file_path, arcname=file_path.relative_to(path))

    # print(f"Successfully created {path} with {num_patients} patients.")
    # print(f"Created flat zip archive at {zip_path}.")

In [5]:
path = "test-data/spatial-v3-test-data.zarr"
generate_synthetic_zarr(path)

/home/klaus/ws/zarr-test-datasets/.venv/lib/python3.12/site-packages/ome_zarr/writer.py:819: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da.to_zarr(


# Validation

In [6]:

store = zarr.open(path, mode="r")
table = store["tables"]["table"]
table_attrs = dict(table.attrs)

print("=== VALIDATION SUMMARY ===")
obs = table["obs"]

# 1) Consolidated metadata
zmetadata_exists = Path("synthetic_study.zarr/.zmetadata").exists() or Path("synthetic_study.zarr/zmetadata").exists()
print(f"Consolidated metadata exists: {zmetadata_exists}")

# 2) Region/image startsWith mapping rule
regions = list(table_attrs["region"])
image_keys = list(store["images"].keys())
violations = [img for img in image_keys if not any(img.startswith(region) for region in regions)]
print(f"images failing startsWith(region): {len(violations)}")
if violations:
    print("Sample violations:", violations[:3])

# 3) Core SpatialData mapping keys
print(f"instance_key: {table_attrs.get('instance_key')}")
print(f"region_key: {table_attrs.get('region_key')}")
print(f"obs has instance_id: {'instance_id' in obs}")
print(f"obs has region: {'region' in obs}")

# 4) Basic consistency counts
print(f"regions in table attrs: {len(regions)}")
print(f"shape keys: {len(list(store['shapes'].keys()))}")
print(f"image keys: {len(image_keys)}")
print("Sample image keys:", image_keys[:3])
print("Sample region keys:", regions[:3])

=== VALIDATION SUMMARY ===
Consolidated metadata exists: False
images failing startsWith(region): 0
instance_key: instance_id
region_key: region
obs has instance_id: True
obs has region: True
regions in table attrs: 12
shape keys: 12
image keys: 12
Sample image keys: ['patient_01_hires', 'patient_02_hires', 'patient_03_hires']
Sample region keys: ['patient_01', 'patient_02', 'patient_03']


In [7]:

store = zarr.open(path, mode="r")
img = store["images"]["patient_01_hires"]["0"][:]

# Estimate nuclei-like dark pixels after overlay.
dark_mask = (img[0] < 90) & (img[1] < 100) & (img[2] < 120)
print("image dtype/shape:", img.dtype, img.shape)
print("dark candidate pixels:", int(dark_mask.sum()))
print("mean pixel value:", round(float(img.mean()), 2))
print("per-channel std:", [round(float(img[c].std()), 2) for c in range(3)])

KeyError: "'0' not found in consolidated metadata."